# Housing Market Relational Database & ETL Pipeline

This project integrates U.S. housing prices, mortgage rates, construction input costs, and bank failures into a relational database for cross-dataset analysis.

The workflow uses Python and Pandas for extraction and transformation, with the original implementation using AWS S3 for source-data storage and AWS RDS for PostgreSQL hosting. SQL was used for relational database design and cross-table analysis.

## Data Sources

All data was acquired through various federal sources, see links below:

Housing Price Index: https://www.fhfa.gov/data/hpi/datasets

Bank failures: https://banks.data.fdic.gov/bankfind-suite/failures

Mortgage rates: https://www.freddiemac.com/pmms

Cement prices: https://fred.stlouisfed.org/series/PCU32733273

Steel prices: https://fred.stlouisfed.org/series/WPU1017

Copper prices: https://fred.stlouisfed.org/series/WPUSIO19011

Lumber prices: https://fred.stlouisfed.org/series/WPU08

# Extract


In [1]:
#import modules
import pandas as pd
import numpy as np

## Load raw data

Raw datasets were originally staged in AWS S3 during the project. For reproducibility, this portfolio version loads the source files from the local data/ directory.

In [ ]:

# Load raw source datasets from local data directory
hpi_og = pd.read_csv("../data/raw/hpi_master.csv")
bank_fails_og = pd.read_csv("../data/raw/bank_failures.csv")
mortgage_rates_og = pd.read_excel("../data/raw/mortgage_rates.xlsx")
concrete_og = pd.read_csv("../data/raw/concrete.csv")
steel_og = pd.read_csv("../data/raw/steel.csv")
copper_og = pd.read_csv("../data/raw/copper.csv")
lumber_og = pd.read_csv("../data/raw/lumber.csv")


## Transform

### Date Dimension

A monthly date dimension was created to provide a shared key for joining datasets with different original time formats and frequencies.

In [4]:
month_id = pd.DataFrame({
    "month": pd.date_range(
        start="1991-01-01",
        end="2025-08-01",
        freq="MS"
    )
})

month_id["date_id"] = range(1, len(month_id) + 1)

month_id.head()

,month,date_id
0,1991-01-01,1
1,1991-02-01,2
2,1991-03-01,3
3,1991-04-01,4
4,1991-05-01,5


### Location Dimension

A location dimension was created to standardize state identifiers and connect state-level data with FHFA regional housing price records.

State abbreviations were mapped to Census divisions, and FHFA place identifiers were incorporated to support joins across geographic levels.

In [5]:
# Create location dimension from state abbreviations and Census divisions

# Load state abbreviations from GitHub
location_id = pd.read_csv(
    "https://raw.githubusercontent.com/jakevdp/data-USstates/refs/heads/master/state-abbrevs.csv"
)

# Add national record
location_id.loc[len(location_id)] = [
    "United States Of America",
    "USA"
]

# Create numeric state identifier
location_id["state_id"] = range(1, len(location_id) + 1)

# Reorder columns
location_id = location_id[
    ["state_id", "state", "abbreviation"]
]

# U.S. Census division mapping
division_map = {
    "CT": "New England Division",
    "ME": "New England Division",
    "MA": "New England Division",
    "NH": "New England Division",
    "RI": "New England Division",
    "VT": "New England Division",

    "NJ": "Middle Atlantic Division",
    "NY": "Middle Atlantic Division",
    "PA": "Middle Atlantic Division",

    "IL": "East North Central Division",
    "IN": "East North Central Division",
    "MI": "East North Central Division",
    "OH": "East North Central Division",
    "WI": "East North Central Division",

    "IA": "West North Central Division",
    "KS": "West North Central Division",
    "MN": "West North Central Division",
    "MO": "West North Central Division",
    "NE": "West North Central Division",
    "ND": "West North Central Division",
    "SD": "West North Central Division",

    "DE": "South Atlantic Division",
    "FL": "South Atlantic Division",
    "GA": "South Atlantic Division",
    "MD": "South Atlantic Division",
    "NC": "South Atlantic Division",
    "SC": "South Atlantic Division",
    "VA": "South Atlantic Division",
    "WV": "South Atlantic Division",
    "DC": "South Atlantic Division",

    "AL": "East South Central Division",
    "KY": "East South Central Division",
    "MS": "East South Central Division",
    "TN": "East South Central Division",

    "AR": "West South Central Division",
    "LA": "West South Central Division",
    "OK": "West South Central Division",
    "TX": "West South Central Division",

    "AZ": "Mountain Division",
    "CO": "Mountain Division",
    "ID": "Mountain Division",
    "MT": "Mountain Division",
    "NV": "Mountain Division",
    "NM": "Mountain Division",
    "UT": "Mountain Division",
    "WY": "Mountain Division",

    "AK": "Pacific Division",
    "CA": "Pacific Division",
    "HI": "Pacific Division",
    "OR": "Pacific Division",
    "WA": "Pacific Division"
}

# Assign Census division to each state
location_id["region"] = (
    location_id["abbreviation"]
    .map(division_map)
)

# Assign national region
location_id.loc[
    location_id["abbreviation"] == "USA",
    "region"
] = "United States"

# Extract FHFA regional identifiers from HPI data
regions_info = (
    hpi_og[["place_name", "place_id"]]
    .drop_duplicates()
)

# Attach FHFA region ID to each state
location_id = pd.merge(
    location_id,
    regions_info,
    left_on="region",
    right_on="place_name",
    how="left"
)

# Rename FHFA place_id
location_id = location_id.rename(
    columns={"place_id": "region_id"}
)

# Remove temporary geographic columns
location_id = location_id.drop(
    columns=["place_name", "region"]
)

# Inspect final location dimension
location_id.head()

,state_id,state,abbreviation,region_id
0,1,Alabama,AL,DV_ESC
1,2,Alaska,AK,DV_PAC
2,3,Arizona,AZ,DV_MT
3,4,Arkansas,AR,DV_WSC
4,5,California,CA,DV_PAC


### Construction Material Cost Index

Construction material price series were standardized to a common January 1991 baseline of 100 so they could be compared directly with the housing price index.

Concrete, steel, copper, and lumber series were converted to monthly datetime values, rebased, and merged into a single material cost table using the shared date dimension.

In [6]:
# Create standardized material cost indexes

def rebase_index(df, date_col, value_col, base_value, output_name):
    material = df.copy()

    material = material.rename(
        columns={
            date_col: "month",
            value_col: "original_index"
        }
    )

    material["month"] = pd.to_datetime(material["month"])

    material[output_name] = round(
        (material["original_index"] / base_value) * 100,
        1
    )

    return material[
        ["month", output_name]
    ]


# Rebase each material series to January 1991 = 100
concrete_index = rebase_index(
    concrete_og,
    "observation_date",
    "PCU327320327320",
    116.5,
    "concrete"
)

steel_index = rebase_index(
    steel_og,
    "observation_date",
    "WPU1017",
    112.2,
    "steel"
)

# Copper data begins in 1990, so restrict to 1991 onward
copper_source = copper_og.copy()
copper_source["observation_date"] = pd.to_datetime(
    copper_source["observation_date"]
)
copper_source = copper_source[
    copper_source["observation_date"] >= "1991-01-01"
]

copper_index = rebase_index(
    copper_source,
    "observation_date",
    "WPUSI019011",
    163.1,
    "copper"
)

lumber_index = rebase_index(
    lumber_og,
    "observation_date",
    "WPU08",
    127.6,
    "lumber"
)

# Merge all material indexes by month
material_costs_index = (
    concrete_index
    .merge(steel_index, on="month", how="left")
    .merge(copper_index, on="month", how="left")
    .merge(lumber_index, on="month", how="left")
)

# Attach shared date key
material_costs_index = pd.merge(
    material_costs_index,
    month_id,
    on="month",
    how="left"
)

# Reorder final table
material_costs_index = material_costs_index[
    [
        "date_id",
        "month",
        "concrete",
        "steel",
        "copper",
        "lumber"
    ]
]

material_costs_index.head()

,date_id,month,concrete,steel,copper,lumber
0,1,1991-01-01,100.0,100.0,100.0,100.0
1,2,1991-02-01,100.2,99.6,97.9,99.7
2,3,1991-03-01,100.6,99.2,98.2,100.2
3,4,1991-04-01,100.3,98.6,97.7,101.3
4,5,1991-05-01,100.3,98.0,95.1,103.7


### Bank Failures by State

FDIC bank failure records were standardized by state and year, then aggregated to create a table containing the number of failures and total reported failure cost for each state-year combination.

State abbreviations were matched to the shared location dimension so the resulting table could be joined with housing and regional data.

In [7]:
# Create bank failure summary by state and year

bank_failures = bank_fails_og.copy()

# Extract state abbreviation from CITYST
bank_failures["state"] = (
    bank_failures["CITYST"]
    .str[-2:]
)

# Attach state_id from location dimension
bank_failures = pd.merge(
    bank_failures,
    location_id[
        ["state_id", "abbreviation"]
    ],
    left_on="state",
    right_on="abbreviation",
    how="left"
)

# Remove records without a matching state
bank_failures = bank_failures[
    bank_failures["state_id"].notna()
].copy()

# Convert failure date to datetime
bank_failures["FAILDATE"] = pd.to_datetime(
    bank_failures["FAILDATE"]
)

# Extract failure year
bank_failures["fail_year"] = (
    bank_failures["FAILDATE"].dt.year
)

# Aggregate failures and cost by state and year
state_failures = (
    bank_failures
    .groupby(
        ["fail_year", "state_id"],
        as_index=False
    )
    .agg(
        fails_by_state=("FAILDATE", "count"),
        cost_per_year=("COST", "sum")
    )
)

# Create primary key
state_failures["id"] = range(
    1,
    len(state_failures) + 1
)

# Reorder columns
state_failures = state_failures[
    [
        "id",
        "state_id",
        "fail_year",
        "fails_by_state",
        "cost_per_year"
    ]
]

state_failures["state_id"] = state_failures["state_id"].astype(int)

state_failures.head()

,id,state_id,fail_year,fails_by_state,cost_per_year
0,1,1,1991,2,88671.0
1,2,3,1991,2,18558.0
2,3,4,1991,2,17218.0
3,4,5,1991,21,2192082.0
4,5,6,1991,4,8584.0


### Mortgage Rates

Freddie Mac Primary Mortgage Market Survey (PMMS) data were cleaned and aggregated from weekly observations to monthly averages. Monthly values were then linked to the shared date dimension for integration with the other housing-market tables.

In [8]:
# Create monthly mortgage rate table
mortgage_rates = mortgage_rates_og.copy()

# Assign descriptive column names
mortgage_rates.columns = [
    "date",
    "thirty_year_frm",
    "thirty_year_fees",
    "fifteen_year_frm",
    "fifteen_year_fees",
    "five_one_arm",
    "five_one_arm_fees",
    "five_one_arm_margin",
    "thirty_yr_to_five_one_arm_spread"
]

# Convert valid dates and remove embedded header rows
mortgage_rates["date"] = pd.to_datetime(
    mortgage_rates["date"],
    errors="coerce"
)

mortgage_rates = mortgage_rates[
    mortgage_rates["date"].notna()
].copy()

# Keep records from 1991 onward
mortgage_rates = mortgage_rates[
    mortgage_rates["date"] >= "1991-01-01"
].copy()

# Convert mortgage fields to numeric
numeric_columns = [
    "thirty_year_frm",
    "thirty_year_fees",
    "fifteen_year_frm",
    "fifteen_year_fees",
    "five_one_arm",
    "five_one_arm_fees",
    "five_one_arm_margin",
    "thirty_yr_to_five_one_arm_spread"
]

mortgage_rates[numeric_columns] = (
    mortgage_rates[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
)

# Create monthly date
mortgage_rates["month"] = (
    mortgage_rates["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Average weekly observations by month
rates_monthly = (
    mortgage_rates
    .groupby("month", as_index=False)[numeric_columns]
    .mean()
    .round(2)
)

# Attach shared date key
rates_monthly = pd.merge(
    rates_monthly,
    month_id,
    on="month",
    how="left"
)

# Reorder columns
rates_monthly = rates_monthly[
    [
        "date_id",
        "month",
        "thirty_year_frm",
        "thirty_year_fees",
        "fifteen_year_frm",
        "fifteen_year_fees",
        "five_one_arm",
        "five_one_arm_fees",
        "five_one_arm_margin",
        "thirty_yr_to_five_one_arm_spread"
    ]
]

rates_monthly.head()

/var/folders/h6/2jffgnzj4q711_d2y51xyr500000gp/T/ipykernel_66150/3589485809.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mortgage_rates["date"] = pd.to_datetime(


,date_id,month,thirty_year_frm,thirty_year_fees,fifteen_year_frm,fifteen_year_fees,five_one_arm,five_one_arm_fees,five_one_arm_margin,thirty_yr_to_five_one_arm_spread
0,1,1991-01-01,9.64,2.05,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1991-02-01,9.36,2.00,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1991-03-01,9.50,2.08,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1991-04-01,9.49,2.02,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1991-05-01,9.47,1.96,NaN,NaN,NaN,NaN,NaN,NaN


### Housing Price Index

FHFA Housing Price Index data were filtered to monthly observations and transformed into the central fact table for the relational database.

Year and period fields were combined into a monthly datetime value, linked to the shared date dimension, and assigned a unique primary key for database loading.


In [9]:
# Create monthly Housing Price Index table

hpi = hpi_og.copy()

# Keep monthly observations only
hpi_monthly = hpi[
    hpi["frequency"].str.lower() == "monthly"
].copy()

# Create monthly datetime field
hpi_monthly["month"] = pd.to_datetime(
    hpi_monthly["yr"].astype(str)
    + "-"
    + hpi_monthly["period"].astype(str).str.zfill(2)
)

# Attach shared date key
hpi_monthly = pd.merge(
    hpi_monthly,
    month_id,
    on="month",
    how="left"
)

# Create primary key
hpi_monthly["hpi_key"] = range(
    1,
    len(hpi_monthly) + 1
)

# Rename year field
hpi_monthly = hpi_monthly.rename(
    columns={"yr": "year"}
)

# Select and reorder columns
hpi_monthly = hpi_monthly[
    [
        "hpi_key",
        "date_id",
        "month",
        "place_id",
        "place_name",
        "index_nsa",
        "index_sa",
        "hpi_type",
        "hpi_flavor",
        "level",
        "year"
    ]
]

hpi_monthly.head()

,hpi_key,date_id,month,place_id,place_name,index_nsa,index_sa,hpi_type,hpi_flavor,level,year
0,1,1,1991-01-01,DV_ENC,East North Central Division,100.00,100.00,traditional,purchase-only,USA or Census Division,1991
1,2,2,1991-02-01,DV_ENC,East North Central Division,100.87,100.87,traditional,purchase-only,USA or Census Division,1991
2,3,3,1991-03-01,DV_ENC,East North Central Division,101.32,100.90,traditional,purchase-only,USA or Census Division,1991
3,4,4,1991-04-01,DV_ENC,East North Central Division,101.73,100.97,traditional,purchase-only,USA or Census Division,1991
4,5,5,1991-05-01,DV_ENC,East North Central Division,102.32,101.30,traditional,purchase-only,USA or Census Division,1991


## Relational Database Schema

The transformed datasets were structured as related tables using shared date and geographic identifiers. The schema below shows the relationships used in the original PostgreSQL database.
![Relational database schema](../docs/database_schema.png)

## Database Load

The original project loaded the transformed tables into a PostgreSQL database hosted on AWS RDS using `psycopg2`.

Because the original cloud database is no longer active, the portfolio version preserves the database connection, table creation, and loading logic below as a commented reproducibility example. Database credentials are supplied through environment variables rather than hard-coded into the notebook.


In [10]:
# ============================================================
# PostgreSQL Database Load
# Original implementation used PostgreSQL hosted on AWS RDS.
# Uncomment and configure environment variables to reproduce.
# ============================================================

# import os
# import psycopg2


# def get_connection():
#     """Create a PostgreSQL database connection."""
#     return psycopg2.connect(
#         host=os.getenv("DB_HOST"),
#         database=os.getenv("DB_NAME"),
#         user=os.getenv("DB_USER"),
#         password=os.getenv("DB_PASSWORD"),
#         port=os.getenv("DB_PORT", "5432")
#     )


# ------------------------------------------------------------
# Create database tables
# ------------------------------------------------------------

# create_tables_sql = """
#
# CREATE TABLE IF NOT EXISTS location_id (
#     state_id INTEGER PRIMARY KEY,
#     state VARCHAR(100),
#     abbreviation VARCHAR(10),
#     region_id VARCHAR(50)
# );
#
# CREATE TABLE IF NOT EXISTS material_costs_index (
#     date_id INTEGER PRIMARY KEY,
#     month DATE,
#     concrete FLOAT,
#     steel FLOAT,
#     copper FLOAT,
#     lumber FLOAT
# );
#
# CREATE TABLE IF NOT EXISTS mortgage_rates (
#     date_id INTEGER PRIMARY KEY,
#     month DATE,
#     thirty_year_frm FLOAT,
#     thirty_year_fees FLOAT,
#     fifteen_year_frm FLOAT,
#     fifteen_year_fees FLOAT,
#     five_one_arm FLOAT,
#     five_one_arm_fees FLOAT,
#     five_one_arm_margin FLOAT,
#     thirty_yr_to_five_one_arm_spread FLOAT
# );
#
# CREATE TABLE IF NOT EXISTS bank_failures_by_state (
#     id INTEGER PRIMARY KEY,
#     state_id INTEGER,
#     fail_year INTEGER,
#     fails_by_state INTEGER,
#     cost_per_year FLOAT,
#     FOREIGN KEY (state_id)
#         REFERENCES location_id(state_id)
# );
#
# CREATE TABLE IF NOT EXISTS housing_price_index (
#     hpi_key INTEGER PRIMARY KEY,
#     date_id INTEGER,
#     month DATE,
#     place_id VARCHAR(50),
#     place_name VARCHAR(225),
#     index_nsa FLOAT,
#     index_sa FLOAT,
#     hpi_type VARCHAR(100),
#     hpi_flavor VARCHAR(100),
#     level VARCHAR(100),
#     year INTEGER
# );
#
# """


# conn = get_connection()
# cur = conn.cursor()
#
# cur.execute(create_tables_sql)
# conn.commit()
#
# cur.close()
# conn.close()


# ------------------------------------------------------------
# Helper function for loading DataFrames
# ------------------------------------------------------------

# def load_dataframe(df, table_name, columns):
#     """
#     Insert a transformed Pandas DataFrame into PostgreSQL.
#     """
#
#     conn = get_connection()
#     cur = conn.cursor()
#
#     placeholders = ", ".join(["%s"] * len(columns))
#     column_names = ", ".join(columns)
#
#     query = f"""
#         INSERT INTO {table_name} ({column_names})
#         VALUES ({placeholders});
#     """
#
#     rows = [
#         tuple(
#             None if pd.isna(value) else value
#             for value in row
#         )
#         for row in df[columns].itertuples(
#             index=False,
#             name=None
#         )
#     ]
#
#     cur.executemany(query, rows)
#     conn.commit()
#
#     cur.close()
#     conn.close()


# ------------------------------------------------------------
# Load transformed tables
# ------------------------------------------------------------

# load_dataframe(
#     location_id,
#     "location_id",
#     [
#         "state_id",
#         "state",
#         "abbreviation",
#         "region_id"
#     ]
# )


# load_dataframe(
#     material_costs_index,
#     "material_costs_index",
#     [
#         "date_id",
#         "month",
#         "concrete",
#         "steel",
#         "copper",
#         "lumber"
#     ]
# )


# load_dataframe(
#     rates_monthly,
#     "mortgage_rates",
#     [
#         "date_id",
#         "month",
#         "thirty_year_frm",
#         "thirty_year_fees",
#         "fifteen_year_frm",
#         "fifteen_year_fees",
#         "five_one_arm",
#         "five_one_arm_fees",
#         "five_one_arm_margin",
#         "thirty_yr_to_five_one_arm_spread"
#     ]
# )


# load_dataframe(
#     state_failures,
#     "bank_failures_by_state",
#     [
#         "id",
#         "state_id",
#         "fail_year",
#         "fails_by_state",
#         "cost_per_year"
#     ]
# )


# load_dataframe(
#     hpi_monthly,
#     "housing_price_index",
#     [
#         "hpi_key",
#         "date_id",
#         "month",
#         "place_id",
#         "place_name",
#         "index_nsa",
#         "index_sa",
#         "hpi_type",
#         "hpi_flavor",
#         "level",
#         "year"
#     ]
# )

## SQL Analysis

The relational database was designed to support both single-table summaries and cross-table analysis of housing prices, mortgage rates, construction costs, and bank failures.

The examples below demonstrate how the shared date and geographic identifiers can be used to query relationships between the datasets.

In [11]:
# ------------------------------------------------------------
# Basic SQL queries
# ------------------------------------------------------------

# Average non-seasonally adjusted HPI by year
# query = """
# SELECT
#     year,
#     AVG(index_nsa) AS avg_hpi
# FROM housing_price_index
# GROUP BY year
# ORDER BY year;
# """


# Highest and lowest observed mortgage rates
# query = """
# SELECT
#     MAX(thirty_year_frm) AS max_30yr,
#     MIN(thirty_year_frm) AS min_30yr,
#     MAX(fifteen_year_frm) AS max_15yr,
#     MIN(fifteen_year_frm) AS min_15yr
# FROM mortgage_rates;
# """


# State with the most bank failures in 2009
# query = """
# SELECT
#     state_id,
#     fails_by_state
# FROM bank_failures_by_state
# WHERE fail_year = 2009
# ORDER BY fails_by_state DESC
# LIMIT 1;
# """


# Maximum construction-material index values
# query = """
# SELECT
#     MAX(concrete) AS max_concrete,
#     MAX(steel) AS max_steel,
#     MAX(copper) AS max_copper,
#     MAX(lumber) AS max_lumber
# FROM material_costs_index;
# """

### Cross-Table Queries

In [12]:
# ------------------------------------------------------------
# Cross-table analytical queries
# ------------------------------------------------------------

# Mortgage rates during the month when concrete costs were lowest
# query = """
# SELECT
#     m.month,
#     m.thirty_year_frm,
#     m.fifteen_year_frm,
#     c.concrete
# FROM mortgage_rates AS m
# JOIN material_costs_index AS c
#     ON m.date_id = c.date_id
# ORDER BY c.concrete ASC
# LIMIT 1;
# """


# Bank failures in states belonging to the Census division
# with the lowest HPI in 2009
# query = """
# SELECT
#     b.state_id,
#     b.fails_by_state,
#     l.state,
#     h.place_name,
#     h.index_nsa
# FROM bank_failures_by_state AS b
# JOIN location_id AS l
#     ON b.state_id = l.state_id
# JOIN housing_price_index AS h
#     ON l.region_id = h.place_id
# WHERE b.fail_year = 2009
#   AND h.year = 2009
#   AND h.index_nsa = (
#       SELECT MIN(index_nsa)
#       FROM housing_price_index
#       WHERE year = 2009
#   );
# """


# Mortgage rate during the month with the highest
# seasonally adjusted HPI
# query = """
# SELECT
#     m.month,
#     m.thirty_year_frm,
#     m.fifteen_year_frm,
#     h.index_sa
# FROM mortgage_rates AS m
# JOIN housing_price_index AS h
#     ON m.date_id = h.date_id
# ORDER BY h.index_sa DESC
# LIMIT 1;
# """


# Average construction-material index during the month
# with the highest seasonally adjusted HPI
# query = """
# SELECT
#     c.month,
#     (
#         c.concrete +
#         c.steel +
#         c.copper +
#         c.lumber
#     ) / 4.0 AS avg_material_index,
#     h.index_sa
# FROM material_costs_index AS c
# JOIN housing_price_index AS h
#     ON c.date_id = h.date_id
# ORDER BY h.index_sa DESC
# LIMIT 1;
# """

## Project Summary

This project demonstrates an end-to-end ETL and relational database workflow integrating multiple U.S. housing and economic datasets.

The pipeline:

- Extracts housing price, mortgage rate, construction material cost, and bank failure data from multiple sources.
- Cleans and standardizes dates, geographic identifiers, and economic indexes using Python and Pandas.
- Rebases construction cost indexes to a common January 1991 baseline for comparison with housing prices.
- Aggregates weekly mortgage rate data into monthly observations.
- Aggregates FDIC bank failures by state and year.
- Creates shared date and geographic dimensions for integrating otherwise independent datasets.
- Structures the transformed data into related tables designed for PostgreSQL.
- Demonstrates SQL queries across housing prices, mortgage rates, construction costs, and bank failures.

The original implementation loaded the transformed tables into a PostgreSQL database hosted on AWS RDS. The portfolio version preserves the database schema and loading logic while keeping the ETL pipeline reproducible without requiring access to the original cloud infrastructure.

### Technologies

Python, Pandas, NumPy, PostgreSQL, SQL, psycopg2, AWS RDS, AWS S3, Jupyter